## До того как вы приступите к решению:
**Tools → Settings → Editor → completions / suggestions / linting → disable**

## Задание 1

Допишите 2 реализации функции `increment()`, которая **увеличивает глобальную переменную `counter` на 1**:

**с/без** (!) python синтаксического сахара. Сигнатуру функции менять нельзя.

**1.1: с python синтаксическим сахаром**

In [ ]:
counter = 0

def increment():
  global counter
  counter += 1

increment()
increment()
assert counter == 2, 'try again'
print(f'{counter=} -- great!')


**1.2: без python синтаксического сахара**

In [ ]:
counter = 0

def increment():
  globals().__setitem__('counter', globals()['counter'] + 1)

increment()
increment()
assert counter == 2, 'try again'
print(f'{counter=} -- great!')


## Задание 2

Достаньте **только функцию `sqrt`** из модуля `math` и исполните sqrt(169).  
Нельзя исполнять `import math`.

Подготовьте 2 решения.

...

In [ ]:
from math import sqrt
result = sqrt(169)

result = __import__('math').sqrt(169)

assert result == 13


## Задание 3

Динамический импорт и перезагрузка.

1. Создайте модуль `mod.py`:

In [ ]:
%%writefile mod.py
msg = "A"


2. Импортируйте его и выведите `msg`

3. Измените `msg` на `B` в файле

4. Без перезагрузки сессии ноутбука, выведите новое значение `msg`

In [ ]:
import importlib
import mod
print(mod.msg)

with open('mod.py', 'w') as f:
    f.write('msg = "B"\n')

importlib.reload(mod)
print(mod.msg)


## Задание 4

У вас есть дирректория `pkg`:

In [ ]:
!mkdir -p pkg

In [ ]:
%%writefile pkg/m1.py
pi = 3.1415_92_65
_e = 2.7
__i = -1

Overwriting pkg/m1.py


```
pkg/
└── m1.py
```

Ниже ячейки для вашего кода, а после задание

### **Первый способ**: через модуль из стандартной библиотеки CPython

In [ ]:
%%writefile pkg/__init__.py
import importlib
_m = importlib.import_module('.m1', package=__name__)
for _n in dir(_m):
    if not _n.startswith('_'):
        globals()[_n] = getattr(_m, _n)


### **Второй способ**: в одну строчку без доп.модулей

In [ ]:
%%writefile pkg/__init__.py
from .m1 import *


### Текст задания:
Нельзя пересоздавать значения `pi`, `_e`, `__i`  и использовать их переменные напрямую в импорте.  
Вам необходимо изменить структуру `pkg` пакета / содержимое его модулей, чтобы следующий код выполнялся корректно:

In [ ]:
from pkg import *
pi

3.14159265

**Важно!**  
При обновлении любых данных в дирректории проекта, вам необходимо перезагружать сессию ipynb:  
`Runtime --> Restart Session` и перезапустить необходимые ячейки задания,  
иначе результаты могут быть для вас некорректными.

## Задание 5

При правильно решённом **задании 4** вам необходимо:
- изменить `pkg`
- дописать код ниже

так, чтобы "дотянуться" до `__i`.  
Нельзя пересоздавать значения `pi`, `_e`, `__i`  и использовать их переменные напрямую в импорте.    
Если вы решите перезагрузить сессию, то для решения **задания 5** необходимо перезапустить ячейки **задания 4**.  

In [ ]:
%%writefile pkg/__init__.py
from . import m1 as _m1
__all__ = [n for n in dir(_m1) if not (n.startswith('__') and n.endswith('__'))]
for _n in __all__:
    globals()[_n] = getattr(_m1, _n)


### Решение

In [ ]:
from pkg import *
__i

-1

## Задание 6

Изменяемое замыкание. Почему этот код ведёт себя неожиданно? Исправьте.

In [ ]:
def create_accumulators():
    accs = []
    totals = []
    for i in range(3):
        totals.append(0)
        def accumulator(x, idx=i):
            totals[idx] += x
            return totals[idx]
        accs.append(accumulator)
    return accs

acc_list = create_accumulators()
print(acc_list[0](10))  # Ожидается 10, но получается ошибка


## Задание 7
При перезапуске сессии ноутбука решение задачи начинается сначала

### 7.1: Востановите работу `print`, не используя `del`

In [ ]:
print = 1

In [ ]:
import builtins
print = builtins.print
print("print восстановлен")


### 7.1: Удалите объект `print`, после востановите его функционал

In [ ]:
print = 1
del print
print("print восстановлен")


## Задание 8

Замыкание с изменяемым состоянием. Создайте функцию-счётчик, которая запоминает количество вызовов между разными экземплярами:

In [ ]:
def make_shared_counter(_state=[0]):
    def counter():
        _state[0] += 1
        return _state[0]
    return counter

c1 = make_shared_counter()
c2 = make_shared_counter()

print(c1())
print(c2())
print(c1())


## Задание 9

Допишите код, чтобы функция `outer` возвращала **словарь с тремя замыканиями**: `add()`, `mul()`, `get()` — работающими с одной и той же закрытой переменной `value`.

In [ ]:
def outer(val=0):
    value = val
    def add(x):
        nonlocal value
        value += x
    def mul(x):
        nonlocal value
        value *= x
    def get():
        return value
    return {'add': add, 'mul': mul, 'get': get}

obj = outer(10)
obj['add'](5)
obj['mul'](2)
assert obj['get']() == 30


## Задание 10

Создать closure, которая принимает функцию и возвращает новую функцию с кэшированием результатов (мемоизацией).

*Теоретическая справка:*

Функция `memoize` принимает другую функцию func и создаёт внутри closure, где есть словарь cache для хранения результатов.

В примере с функцией `fib` (числа Фибоначчи) мемоизация уменьшает количество рекурсивных вызовов с экспоненциального до линейного, так как результаты для каждого n вычисляются один раз.

Таким образом, мемоизация экономит время за счёт памяти — класическая оптимизация "время против памяти" — и особенно полезна для функций с дорогими вычислениями и повторяющимися входами.

Алгоритм для решения:

- Функция memoize принимает другую функцию func и создаёт внутри closure, где есть словарь cache для хранения результатов.

- Внутренняя функция wrapper проверяет, есть ли для данного входного аргумента (x) уже вычисленный результат в словаре cache.

- Если результат есть, то он возвращается из кеша, и вычисления не повторяются.

- Если нет, то вызывается исходная функция func(x), результат сохраняется в cache и возвращается.


In [ ]:
def memoize(func):
    cache = {}
    def wrapper(x):
        if x not in cache:
            cache[x] = func(x)
        return cache[x]
    return wrapper

@memoize
def fib(n):
    if n < 2:
        return n
    return fib(n-1) + fib(n-2)

print(fib(10))  # 55
